In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, lead, lag, count, avg

spark = (SparkSession.builder
         .appName("apply-window-functions")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [0]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles.csv"))

In [0]:
df = df.filter(col('country').isNotNull() & col('date_added').isNotNull())

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("country").orderBy("date_added")

In [0]:
# Assign row numbers within each partition
result = df.withColumn("row_number", row_number().over(window_spec))
result.select("title","country","date_added","row_number").show()

In [0]:
# Add lead column
df = df.withColumn("lead_date_added", lead("date_added").over(window_spec))
# Add lag column
df = df.withColumn("lag_date_added", lag("date_added").over(window_spec))

df.select("title","country","date_added","lead_date_added","lag_date_added").show(3)

### Nested Window Functions

In [0]:
from pyspark.sql.functions import sum, lead
from pyspark.sql.window import Window

window_spec = Window.partitionBy("country").orderBy("release_year")
df = df.withColumn("running_total", count("show_id").over(window_spec))
df = df.withColumn("next_running_total", lead("running_total").over(window_spec))
df = df.withColumn("diff", df["next_running_total"] - df["running_total"])

### Window Frames

In [0]:
data = [(1, 10), (2, 15), (3, 20), (4, 25), (5, 30)]
df = spark.createDataFrame(data, ["id", "value"])

windowSpec = Window.orderBy("id").rowsBetween(-2, 0)
df = df.withColumn("rolling_avg", avg(df["value"]).over(windowSpec))

df.show()

In [0]:
spark.stop()